In [4]:
# ============================================================
# POC: Predictor de Inflación Cloud-Native
# MINE4105 - Soluciones de Datos en la Nube
# Celda 1: Setup e instalación de dependencias
# ============================================================

import subprocess
subprocess.run(['pip', 'install', 'scikit-learn', 'pyarrow', '--upgrade', '-q'])

import boto3
import joblib
import pandas as pd
import numpy as np
import time
import json
import io

print("Dependencias instaladas")
print(f"boto3 version: {boto3.__version__}")

Dependencias instaladas
boto3 version: 1.43.6


In [5]:
# ============================================================
# Celda 2: Conexión a S3 - Carga de modelo y datos
# ============================================================

BUCKET = 'poc-inflation-predictor'
REGION = 'us-east-1'

s3 = boto3.client('s3', region_name=REGION)

# Cargar modelo Random Forest h1 desde S3
print("Descargando modelo desde S3...")
t0 = time.time()
model_obj = s3.get_object(Bucket=BUCKET, Key='models/best_RandomForest_h1.joblib')
model = joblib.load(io.BytesIO(model_obj['Body'].read()))
print(f"Modelo cargado en {time.time()-t0:.2f}s")

# Cargar datos desde S3
print("Descargando datos desde S3...")
data_obj = s3.get_object(Bucket=BUCKET, Key='gold/features.parquet')
df = pd.read_parquet(io.BytesIO(data_obj['Body'].read()))
print(f"Datos cargados: {df.shape[0]} filas, {df.shape[1]} columnas")
print(df.tail(3))

Descargando modelo desde S3...
Modelo cargado en 0.11s
Descargando datos desde S3...
Datos cargados: 395 filas, 124 columnas
           date  cpi_lag1  cpi_lag3  cpi_lag6  cpi_lag12  fed_rate_lag1  \
392  2025-10-01   324.245   322.169   320.302    315.631           4.22   
393  2025-11-01   324.245   323.291   320.620    316.528           4.09   
394  2025-12-01   325.063   324.245   321.435    317.604           3.88   

     fed_rate_lag3  fed_rate_lag6  fed_rate_lag12  oil_price_lag1  ...  \
392           4.33           4.33            4.83       63.959048  ...   
393           4.33           4.33            4.64       60.894545  ...   
394           4.22           4.33            4.48       60.062222  ...   

     quarter_3  quarter_4  inflation_mom_lag1  inflation_mom_lag3  \
392          0          1            0.295090            0.228351   
393          0          1            0.000000            0.348264   
394          0          1            0.252278            0.295090   



In [6]:
# ============================================================
# Celda 3: Inferencia y medición de latencia
# ============================================================

# Usar última fila disponible como input de predicción
FEATURE_COLS = [c for c in df.columns if c not in ['date', 'inflation_mom']]
X_last = df[FEATURE_COLS].iloc[[-1]]

print(f"Fecha de predicción: {df['date'].iloc[-1]}")
print(f"Features usados: {len(FEATURE_COLS)}")

# Medir latencia de inferencia
latencias = []
N = 100  # 100 predicciones para calcular p50/p95

for i in range(N):
    t0 = time.time()
    pred = model.predict(X_last)[0]
    latencias.append((time.time() - t0) * 1000)  # en ms

p50 = np.percentile(latencias, 50)
p95 = np.percentile(latencias, 95)

print(f"\n Predicción CPI MoM h1: {pred:.4f}%")
print(f"\n--- Latencias ML ({N} llamadas) ---")
print(f"  p50: {p50:.2f} ms")
print(f"  p95: {p95:.2f} ms")
print(f"  Criterio p95 < 2000ms: {'CUMPLE' if p95 < 2000 else 'NO CUMPLE'}")

Fecha de predicción: 2025-12-01
Features usados: 122

 Predicción CPI MoM h1: 0.2305%

--- Latencias ML (100 llamadas) ---
  p50: 45.93 ms
  p95: 47.59 ms
  Criterio p95 < 2000ms: CUMPLE


In [ ]:
# ============================================================
# Celda 4: Narrativa ejecutiva — Google Gemini API
# API key leída desde AWS Secrets Manager (nunca en texto plano)
# ============================================================

import subprocess
subprocess.run(['pip', 'install', 'google-genai', '-q'], capture_output=True)
from google import genai
import json as _json

# Leer API key desde Secrets Manager
secrets = boto3.client('secretsmanager', region_name='us-east-1')
secret = secrets.get_secret_value(SecretId='poc-inflation/gemini-api-key')
gemini_api_key = _json.loads(secret['SecretString'])['api_key']

client_llm = genai.Client(api_key=gemini_api_key)

prompt = f"""Eres un analista macroeconómico senior.
El modelo de Machine Learning predice una inflación mensual (CPI MoM) de {pred:.4f}% para enero 2026.
Contexto: Fed Rate actual ~3.88%, precio del petróleo ~$60 USD, tendencia reciente de desaceleración.
En máximo 3 oraciones, genera una narrativa ejecutiva para un Portfolio Manager."""

t0 = time.time()

response = client_llm.models.generate_content(
    model='gemini-2.5-flash',
    contents=prompt,
    config=genai.types.GenerateContentConfig(
        max_output_tokens=200,
        thinking_config=genai.types.ThinkingConfig(thinking_budget=0)
    )
)

latencia_llm = (time.time() - t0) * 1000
narrativa = response.text

latencia_e2e = p95 + latencia_llm

print(f"Latencia LLM (Gemini): {latencia_llm:.0f} ms")
print(f"Latencia e2e total (ML p95 + LLM): {latencia_e2e:.0f} ms")
print(f"Criterio e2e < 5000ms: {'CUMPLE' if latencia_e2e < 5000 else 'NO CUMPLE'}")
print(f"\n--- Narrativa ejecutiva ---\n{narrativa}")